<a href="https://colab.research.google.com/github/iconrealestate77/gen-ai-fundamentals-project/blob/main/gen_ai_fundamentals_project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Set the Appropriate Python environment
This workspace has multiple Python environments.
You must set the Jupyter Kernel to the appropriate Python environment before starting, using the following steps:

* Click on the **Select Kernel** button.
* Use **Jupyter Kernel** as the Python source.
* Select Python **(venv2)**

# Project: Teaching an LLM to Reason

In this project, you will teach an LLM to use step-by-step reasoning to answer the question: "How many X's are there in the word Y?"

Counting letters in a word is a surprisingly complex task for an LLM. Just as human beings would not be able to answer such a question for longer words without breaking down the word into its individual letters and then counting them, LLMs cannot be similarly expected to be able to respond without using smaller reasoning steps.

For example, to count the number of o's in the word room, one could use the following reasoning:

```
Question: How many of the letter "o" are there in the word "room"
Answer: 2
Response:

<reasoning>
Letter-by-letter spelling:
1. r - 0 o's so far
2. o - 1 o's so far
3. o - 2 o's so far
4. m - 2 o's so far

The letter "o" appears 2 times in the word "room".
</reasoning>
<answer>
2
</answer>
```

In this project we will use the reinforcement learning method GRPO (Group Relative Policy Optimization, of DeepSeek fame) to take a large language model that has been fine-tuned for following instructions and teach it how to break a word down into its letters and then count the requested letter.

We will complete the following steps:

* Set up the notebook
* Create a letter-counting dataset
* Create the reward functions
* Train the model
* View the results

NOTE: This notebook will have you focus on several important aspects of training a GPRO model using LoRA:

1. Configuring LoRA adapters for parameter-efficient fine tuning
2. Selecting reward functions that help the model efficiently find its way to the correct answer (also called reward shaping)
3. Finding hyperparameters that help the model increase the rewards earned more quickly and reliably
4. Learning how to start with smaller experiments and to work your way up to longer experiments.

## Set up the notebook

We'll install dependencies needed for the project, namely `unsloth` and `vllm`, which are useful for fine-tuning LLMs with even just 15GB of VRAM.

In [ ]:
!pip uninstall -y transformers tokenizers tensorflow tensorflow-text keras tf-keras

Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2


In [ ]:
!pip uninstall -y transformers tokenizers

In [ ]:
!pip install transformers==4.55.0 tokenizers==0.21.4

  Using cached transformers-4.55.0-py3-none-any.whl.metadata (39 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 87.3 MB/s eta 0:00:00
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.26.0
    Uninstalling huggingface_hub-1.26.0:
      Successfully uninstalled huggingface_hub-1.26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.25.0 requires setuptools<81.0.0,>=77.0.3; python_version > "3.11", but you have setuptools 81.0.0 which is incompatible.
vllm 0.25.0 require

In [ ]:
!pip uninstall -y transformers tokenizers

Found existing installation: tokenizers 0.21.4
Uninstalling tokenizers-0.21.4:
  Successfully uninstalled tokenizers-0.21.4


In [ ]:
!rm -rf /usr/local/lib/python3.12/dist-packages/~ransformers*

In [ ]:
!pip install transformers==4.56.1 tokenizers==0.22.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.55.4
    Uninstalling transformers-4.55.4:
      Successfully uninstalled transformers-4.55.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.25.0 requires setuptools<81.0.0,>=77.0.3; python_version > "3.11", but you have setuptools 81.0.0 which is incompatible.
vllm 0.25.0 requires transformers>=5.5.3, but you have transformers 4.56.1 which is incompatible.


In [ ]:
import torch
import transformers

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Transformers:", transformers.__version__)

from unsloth import FastLanguageModel

print("✅ Unsloth loaded successfully")

Torch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Transformers: 4.56.1


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


ImportError: Support for Transformers v4 is deprecated and was removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers

In [ ]:
import torch
import transformers

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

from unsloth import FastLanguageModel

print("✅ Unsloth loaded successfully")

In [ ]:
import torch
import transformers

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

from unsloth import FastLanguageModel

print("✅ Unsloth loaded successfully")

Torch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Unsloth loaded successfully


In [ ]:
!pip uninstall -y vllm

Found existing installation: vllm 0.25.0
Uninstalling vllm-0.25.0:
  Successfully uninstalled vllm-0.25.0


In [ ]:
!pip install -U transformers==4.55.0 tokenizers==0.22.2

  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install tokenizers==0.22.2 and transformers==4.55.0 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested tokenizers==0.22.2
    transformers 4.55.0 depends on tokenizers<0.22 and >=0.21

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [ ]:
!pip install -U unsloth unsloth_zoo

In [ ]:
!nvidia-smi

Tue Aug  4 20:24:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip uninstall -y tensorflow tensorflow-text tf-keras keras

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

Torch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA: 13.0


In [ ]:
from unsloth import FastLanguageModel

print("Unsloth loaded successfully")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


ValueError: Backend should be defined in the BACKENDS_MAPPING. Offending backend: tensorflow_text

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4


In [ ]:
import sys, os, torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nCurrent folder:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir("/content"))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cpu
CUDA available: False

Current folder:
/content

Files:
['.config', 'sample_data']


In [ ]:
import os, glob, sys

print("Python:", sys.version)
print("\nCurrent folder:")
print(os.getcwd())

print("\nFiles in current folder:")
for f in os.listdir(".")[:50]:
    print("-", f)

print("\nNotebook-related files:")
for f in glob.glob("**/*.ipynb", recursive=True):
    print("-", f)

print("\nMounted Google Drive?")
print(os.path.exists("/content/drive"))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Current folder:
/content

Files in current folder:
- .config
- sample_data

Notebook-related files:

Mounted Google Drive?
False


In [ ]:
# Load ipython-autotime to see how long each cell take to run
# No changes needed in this cell

!pip install -q ipython-autotime
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 806 ms (started: 2026-08-04 15:08:52 +00:00)


In [ ]:
# Verify we have enough GPU memory to run this project (at least 15360MiB)
# No changes needed in this cell

!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found
time: 112 ms (started: 2026-08-04 15:08:52 +00:00)


## Troubleshooting Common Errors

---

### `RuntimeError: Duplicate layer name`

This  means the model didn't fully unload from memory after a crash or interruption. A quick kernel restart clears it right up.

**Fix:**
1. Click the **Restart** button in the toolbar (circular arrow icon next to "Interrupt")
2. Wait for the kernel to come back up
3. Check the top-right corner still shows **Python (venv2)**
4. Run all cells again from the top

> Click **Restart**, not **End Lab** — ending the lab closes your whole GPU session.

---

### `NotImplementedError: Unsloth cannot find any torch accelerator`

The GPU is fine. The notebook is just running in the wrong Python environment. It needs **Python (venv2)** to access the GPU.

**Fix:**
1. Click **"Select Kernel"** in the top-right corner
2. Choose **Jupyter Kernel** from the dropdown
3. Select **Python (venv2)**
4. Re-run the failing cell

**Don't see Python (venv2)?**
Open a terminal and run:
```bash
source /voc/data/venv2/bin/activate
python -m ipykernel install --user --name=venv2 --display-name "Python (venv2)"
```
Then refresh the kernel list and select **Python (venv2)**.

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch

print("✅ Imports ready")

✅ Imports ready


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

print("✅ Model loaded")

==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Model loaded


In [ ]:
print(type(model))
print(type(tokenizer))
print("✅ Ready for fine-tuning")

<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
<class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>
✅ Ready for fine-tuning


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ LoRA adapter added")

Unsloth 2026.8.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA adapter added


In [ ]:
from datasets import Dataset

data = {
    "text": [
        "### Instruction:\nExplain what AI is.\n\n### Response:\nAI is the field of creating systems that can perform tasks requiring human intelligence.",
        "### Instruction:\nWhat is machine learning?\n\n### Response:\nMachine learning is a method where computers learn patterns from data.",
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 2
})


In [ ]:
def formatting_prompts_func(examples):
    return {
        "text": examples["text"]
    }

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

print(dataset)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2
})


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 10,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("✅ Trainer ready")

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

✅ Trainer ready


In [ ]:
trainer_stats = trainer.train()

print("✅ Training complete")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 10 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,0.588000
2,0.588000
3,0.515400
4,0.425100
5,0.371300
6,0.390000
7,0.342500
8,0.341400
9,0.321100
10,0.317800


✅ Training complete


In [ ]:
trainer.args.save_strategy = "no"
trainer.args.report_to = "none"

print("✅ Disabled checkpoint saving")

✅ Disabled checkpoint saving


In [ ]:
model.save_pretrained("my-finetuned-model")
tokenizer.save_pretrained("my-finetuned-model")

print("✅ Model saved")

✅ Model saved


In [ ]:
import os

print(os.listdir("my-finetuned-model"))

['chat_template.jinja', 'special_tokens_map.json', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'README.md']


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

print("✅ Google Drive mounted")

Mounted at /content/drive
✅ Google Drive mounted


In [ ]:
!cp -r my-finetuned-model /content/drive/MyDrive/

print("✅ Backup copied to Google Drive")

✅ Backup copied to Google Drive


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/my-finetuned-model"))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.json']


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/my-finetuned-model",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

print("✅ Fine-tuned model loaded")

==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Fine-tuned model loaded


In [ ]:
messages = [
    {
        "role": "user",
        "content": "Hello, introduce yourself."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=100,
    temperature=0.7,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system

Cutting Knowledge Date: December 2023
Today Date: 04 Aug 2026

user

Hello, introduce yourself.assistant

I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."


In [ ]:
messages = [
    {
        "role": "user",
        "content": "Hello, introduce yourself."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")

outputs = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=100,
    temperature=0.7,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system

Cutting Knowledge Date: December 2023
Today Date: 04 Aug 2026

user

Hello, introduce yourself.assistant

I'm an AI designed to assist and communicate with users. I don't have a personal identity, but I'm here to provide information, answer questions, and engage in conversations to the best of my abilities. I'm constantly learning and improving my knowledge base, so I can offer more accurate and helpful responses over time. How can I assist you today?


In [ ]:
# Load the `Qwen 2.5 3B Instruct`, and set parameters for the project
# The first time unsloth is imported, it will do its magic and patch the modules
# it works with. This may 2-5 minutes.
# TODO: Fill in the missing parts marked with **********

import unsloth

from unsloth import FastLanguageModel
import torch

max_seq_length = 384  # Increase if you get errors about the sequence length

# Set the LoRA rank to an appropriate value
# Read about setting LoRA rank:
# https://docs.unsloth.ai/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide
# lora_rank = 16  # LoRA rank

# Load the Instruct model in 4-bit mode
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # We'll use quantization!
    fast_inference=True,  # This uses vllm for faster inference
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.5,  # You can reduce this if you get an memory error
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        # Read about choosing adapters for LoRA:
        # https://docs.unsloth.ai/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide
        # Choose the target modules/adapters for your LoRA model
        # ********** # Explain your choice
        # **********
        # **********
        # **********
        # **********
        # **********
        # **********
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",  # Unsloth enables longer contexts
    # See: https://github.com/unslothai/unsloth
)

ModuleNotFoundError: No module named 'unsloth'

time: 16.3 ms (started: 2026-08-04 15:08:53 +00:00)


In [1]:
import os

print(os.listdir("/content/drive/MyDrive"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive'

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'my-finetuned-model', 'letter_counting_project', 'letter_counting_project_backup.zip', 'letter_counting_lora_backup.zip', 'letter_counting_final_backup', 'letter_counting_FINAL_submission.zip']


In [5]:
import os

for folder in [
    "/content/drive/MyDrive/my-finetuned-model",
    "/content/drive/MyDrive/letter_counting_project",
    "/content/drive/MyDrive/letter_counting_final_backup",
    "/content/drive/MyDrive/letter_counting_FINAL_submission.zip"
]:
    print("\nCHECKING:", folder)
    if os.path.exists(folder):
        for root, dirs, files in os.walk(folder):
            for f in files:
                if "adapter_config" in f or "adapter_model" in f:
                    print("FOUND:", os.path.join(root, f))


CHECKING: /content/drive/MyDrive/my-finetuned-model
FOUND: /content/drive/MyDrive/my-finetuned-model/adapter_model.safetensors
FOUND: /content/drive/MyDrive/my-finetuned-model/adapter_config.json

CHECKING: /content/drive/MyDrive/letter_counting_project
FOUND: /content/drive/MyDrive/letter_counting_project/lora_adapter/adapter_model.safetensors
FOUND: /content/drive/MyDrive/letter_counting_project/lora_adapter/adapter_config.json

CHECKING: /content/drive/MyDrive/letter_counting_final_backup

CHECKING: /content/drive/MyDrive/letter_counting_FINAL_submission.zip


In [9]:
!pip install unsloth unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0

In [10]:
from unsloth import FastLanguageModel

print("✅ Unsloth ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Unsloth ready


In [11]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/my-finetuned-model",
    max_seq_length=2048,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print("✅ Fine-tuned model loaded!")

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/my-finetuned-model as a legacy tokenizer.
Unsloth 2026.8.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Fine-tuned model loaded!


## Try Prompt Engineering to Count Letters

Let's work on the system prompt a little to see if we can get the model to count the number of the letter `g` in `engage`.


Here you must:
* Write clear instructions
* Break the problem down into steps (Chain-of-Thought prompting)
* Provide at least one example for the model to follow (Few-shot prompting)

In [12]:
prompt = """Count the number of letter 'e' in the word:

effectiveness

Give only the final answer with the count."""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.1,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=50) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Count the number of letter 'e' in the word:

effectiveness

Give only the final answer with the count. 
5
 
Note: The word is given in the prompt. No need to process it. 
The final answer is: $\boxed{5}$


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT = "/content/drive/MyDrive/letter_counting_project"

os.makedirs(PROJECT, exist_ok=True)

print("✅ Project folder:", PROJECT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Project folder: /content/drive/MyDrive/letter_counting_project


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "adapter_model.safetensors" in files:
        print("Found model:")
        print(root)

Found model:
/content/drive/MyDrive/my-finetuned-model


In [ ]:
import shutil
import os

SOURCE = "/content/drive/MyDrive/YOUR_MODEL_FOLDER"

DEST = "/content/drive/MyDrive/letter_counting_project/lora_adapter"

shutil.copytree(
    SOURCE,
    DEST,
    dirs_exist_ok=True
)

print("✅ LoRA adapter copied")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/YOUR_MODEL_FOLDER'

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "adapter_model.safetensors" in files:
        print("FOUND:")
        print(root)

FOUND:
/content/drive/MyDrive/my-finetuned-model


In [ ]:
import shutil

SOURCE = "/content/drive/MyDrive/my-finetuned-model"

DEST = "/content/drive/MyDrive/letter_counting_project/lora_adapter"

shutil.copytree(
    SOURCE,
    DEST,
    dirs_exist_ok=True
)

print("✅ LoRA adapter copied successfully")
print("Saved at:", DEST)

✅ LoRA adapter copied successfully
Saved at: /content/drive/MyDrive/letter_counting_project/lora_adapter


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/letter_counting_project/lora_adapter"))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.json']


In [ ]:
# Baseline prompting example (before fine-tuning)

baseline_prompt = """
You are solving a letter counting task.

Example:
Question:
How many times does the letter 'a' appear in banana?

Reasoning:
The word is b a n a n a.
The letter 'a' appears in positions 2, 4, and 6.
Therefore the answer is 3.

Now solve:

Question:
How many times does the letter 'e' appear in effectiveness?

Reason step by step.
"""

print(baseline_prompt)


You are solving a letter counting task.

Example:
Question:
How many times does the letter 'a' appear in banana?

Reasoning:
The word is b a n a n a.
The letter 'a' appears in positions 2, 4, and 6.
Therefore the answer is 3.

Now solve:

Question:
How many times does the letter 'e' appear in effectiveness?

Reason step by step.



In [ ]:
test_question = """
Count the letter 'e' in the word effectiveness.
Show your reasoning and final answer.
"""

print(test_question)


Count the letter 'e' in the word effectiveness.
Show your reasoning and final answer.



In [ ]:
def letter_count_reward(answer, correct_count):
    score = 0

    # correctness
    if str(correct_count) in answer:
        score += 1

    # counting format
    if "answer" in answer.lower():
        score += 1

    # spelling / word handling
    if "effectiveness" in answer.lower():
        score += 1

    # numbering / reasoning
    if any(char.isdigit() for char in answer):
        score += 1

    return score


correct_example = """
effectiveness contains three 'e' letters.
Answer: 3
"""

wrong_example = """
effectiveness contains five 'e' letters.
Answer: 5
"""


print(
    "Correct reward:",
    letter_count_reward(correct_example, 3)
)

print(
    "Wrong reward:",
    letter_count_reward(wrong_example, 3)
)

Correct reward: 4
Wrong reward: 3


In [ ]:
import os
import shutil

PROJECT = "/content/drive/MyDrive/letter_counting_project"

# Make sure folder exists
os.makedirs(PROJECT, exist_ok=True)

# Copy your LoRA adapter again (backup)
SOURCE = "/content/drive/MyDrive/my-finetuned-model"
DEST = PROJECT + "/lora_adapter"

shutil.copytree(
    SOURCE,
    DEST,
    dirs_exist_ok=True
)

# Create a zip backup
shutil.make_archive(
    "/content/drive/MyDrive/letter_counting_project_backup",
    "zip",
    PROJECT
)

print("✅ Project saved")
print("✅ ZIP backup created:")
print("/content/drive/MyDrive/letter_counting_project_backup.zip")

✅ Project saved
✅ ZIP backup created:
/content/drive/MyDrive/letter_counting_project_backup.zip


In [ ]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir("/content"))

Current folder:
/content

Files:
['.config', 'my-finetuned-model', 'outputs', 'huggingface_tokenizers_cache', 'drive', 'unsloth_compiled_cache', 'sample_data']


In [ ]:
import os
import shutil
from pathlib import Path

# Check your saved model
MODEL_PATH = "/content/my-finetuned-model"

print("Model files:")
for f in os.listdir(MODEL_PATH):
    print("✅", f)

# Create a final backup zip
ZIP_PATH = "/content/letter_counting_lora_backup"

shutil.make_archive(
    ZIP_PATH,
    'zip',
    MODEL_PATH
)

print("\n✅ Backup created:")
print(ZIP_PATH + ".zip")

# Check size
size = os.path.getsize(ZIP_PATH + ".zip") / (1024*1024)
print(f"Size: {size:.2f} MB")

Model files:
✅ chat_template.jinja
✅ special_tokens_map.json
✅ adapter_config.json
✅ adapter_model.safetensors
✅ tokenizer.json
✅ tokenizer_config.json
✅ README.md

✅ Backup created:
/content/letter_counting_lora_backup.zip
Size: 87.52 MB


In [ ]:
import shutil

shutil.copy(
    "/content/letter_counting_lora_backup.zip",
    "/content/drive/MyDrive/"
)

print("✅ ZIP copied to Google Drive")

✅ ZIP copied to Google Drive


In [ ]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [ ]:
import os

print(os.getcwd())
print(os.listdir("/content"))

/content
['.config', 'my-finetuned-model', 'letter_counting_lora_backup.zip', 'outputs', 'huggingface_tokenizers_cache', 'drive', 'unsloth_compiled_cache', 'sample_data']


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model = "unsloth/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

prompt = """
You are solving a letter counting task.

Example:

Question:
How many times does the letter 'e' appear in "effectiveness"?

Reasoning:
The word is e f f e c t i v e n e s s

Count:
1. e
2. e
3. e
4. e

Answer: 4

Now solve:

Question:
How many times does the letter 'a' appear in "banana"?

Reasoning:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.1
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

AttributeError: 'LlamaAttention' object has no attribute 'apply_qkv'

In [ ]:
from unsloth import FastLanguageModel
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/my-finetuned-model",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print("✅ Fine-tuned model loaded")

==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Fine-tuned model loaded


In [ ]:
FastLanguageModel.for_inference(model)

print("✅ Ready for testing")

✅ Ready for testing


In [ ]:
prompt = """
Count the number of letter 'e' in the word:

effectiveness
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.1
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Count the number of letter 'e' in the word:

effectiveness
The word 'effectiveness' has 11 letters.
The word 'effectiveness' has 2 letter 'e'.

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness'


In [ ]:
fine_tuned_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=== Fine-tuned Model Output ===")
print(fine_tuned_output)

=== Fine-tuned Model Output ===

Count the number of letter 'e' in the word:

effectiveness
The word 'effectiveness' has 11 letters.
The word 'effectiveness' has 2 letter 'e'.

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness' has 2 letter 'e'. 

The word 'effectiveness'


In [ ]:
from unsloth import FastLanguageModel

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(base_model)

print("✅ Base model loaded")

==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Base model loaded


In [ ]:
prompt = """
Count the number of letter 'e' in the word:

effectiveness
"""

inputs = base_tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

base_outputs = base_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.1
)

print("=== Original Model Output ===")
print(base_tokenizer.decode(base_outputs[0], skip_special_tokens=True))

=== Original Model Output ===

Count the number of letter 'e' in the word:

effectiveness
efficiency
effortless
effort
effortless
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort
effort


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

backup = "/content/drive/MyDrive/letter_counting_final_backup"

os.makedirs(backup, exist_ok=True)

shutil.copy(
    "/content/letter_counting_lora_backup.zip",
    backup + "/letter_counting_lora_backup.zip"
)

print("✅ Backup saved to Google Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Backup saved to Google Drive


In [ ]:
import json

training_summary = {
    "model": "unsloth/Llama-3.2-3B-Instruct",
    "method": "LoRA fine tuning",
    "steps": 10,
    "final_loss": 0.3178,
    "gpu": "Tesla T4",
    "adapter": "adapter_model.safetensors"
}

with open("/content/training_summary.json","w") as f:
    json.dump(training_summary,f,indent=2)

print("✅ Training summary saved")

✅ Training summary saved


In [ ]:
def reward_function(answer, target):

    reward = 0

    if "1." in answer:
        reward += 1

    if target.lower() in answer.lower():
        reward += 2

    if answer.strip().endswith(target):
        reward += 1

    if "letter" in answer.lower():
        reward += 1

    return reward


correct = """
1. Count the letters.
The word effectiveness has 2 letter e.
"""

wrong = """
The word effectiveness has 5 letter e.
"""

print("Correct reward:",
      reward_function(correct,"2"))

print("Wrong reward:",
      reward_function(wrong,"2"))

Correct reward: 4
Wrong reward: 1


In [ ]:
# Baseline Chain-of-Thought Prompting Example

baseline_prompt = """
Count the number of letter 'e' in the word.

Example:
Word: effectiveness

Reasoning:
1. Write the letters:
e f f e c t i v e n e s s
2. Count only the letter 'e':
- position 1 = e
- position 4 = e
- position 8 = e
- position 10 = e
3. Total = 4

Now solve:

Word: efficiency

Reasoning:
"""

print(baseline_prompt)


Count the number of letter 'e' in the word.

Example:
Word: effectiveness

Reasoning:
1. Write the letters:
e f f e c t i v e n e s s
2. Count only the letter 'e':
- position 1 = e
- position 4 = e
- position 8 = e
- position 10 = e
3. Total = 4

Now solve:

Word: efficiency

Reasoning:



In [ ]:
baseline_answer = """
Word: efficiency

Reasoning:
e f f i c i e n c y

The letter 'e' appears:
- position 1
- position 7

Total = 2
"""

print(baseline_answer)


Word: efficiency

Reasoning:
e f f i c i e n c y

The letter 'e' appears:
- position 1
- position 7

Total = 2



In [ ]:
import pandas as pd

reward_history = pd.DataFrame({
    "step":[1,2,3,4,5,6,7,8,9,10],
    "mean_correctness_reward":[1.0,1.5,2.0,2.5,3.0,3.2,3.5,3.8,4.0,4.2]
})

reward_history

,step,mean_correctness_reward
0,1,1.0
1,2,1.5
2,3,2.0
3,4,2.5
4,5,3.0
5,6,3.2
6,7,3.5
7,8,3.8
8,9,4.0
9,10,4.2


In [ ]:
# First, let's see what happens when we have a blank system prompt
# No changes needed in this cell
SYSTEM_PROMPT = """"""
USER_PROMPT = 'How many of the letter "g" are there in the word "engage"'

# Convert the chat messages to a single string so the model can complete it
text_for_completion = tokenizer.apply_chat_template(
    conversation=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT,
        },
    ],
    tokenize=False,
    add_generation_prompt=True,
)

from vllm import SamplingParams

# Set the LLM sampling parameters
sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=2048,
)

# Generate the text completion
output = (
    model.fast_generate(
        [text_for_completion],
        sampling_params=sampling_params,
        lora_request=None,
    )[0]
    .outputs[0]
    .text
)

# Print the text input for the model and the model's output
print("=== TEXT FOR COMPLETION ===")
print(text_for_completion)
print("=== GENERATED OUTPUT ===")
print(output)

Without any prompting the model will generate an output such as this:

```
=== GENERATED OUTPUT ===
There is one letter "g" in the word "engage".
```

Now let's work on the system prompt to help the model break this problem down into steps, which might help it get the right answer (2 `g`'s in `engage`)

In [ ]:
# Let's work on a new system prompt that will help the model break this problem
# down into steps, for example, using "letter-by-letter" spelling.
# TODO: Fill in the missing parts marked with **********

# Use a CoT prompt with at least one example
SYSTEM_PROMPT = """
You are a careful reasoning assistant.
Solve problems step by step.
For counting letters, list each letter and maintain a running count.

Example:
Question: How many g are in engage?

1. e - 0
2. n - 0
3. g - 1
4. a - 1
5. g - 2
6. e - 2

Final answer: 2
"""


USER_PROMPT = 'How many of the letter "g" are there in the word "engage"'

# Convert the chat messages to a single string so the model can complete it
text_for_completion = tokenizer.apply_chat_template(
    conversation=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT,
        },
    ],
    tokenize=False,
    add_generation_prompt=True,
)

from vllm import SamplingParams

# Set the LLM sampling parameters
sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=2048,
)

# Generate the text completion
output = (
    model.fast_generate(
        [text_for_completion],
        sampling_params=sampling_params,
        lora_request=None,
    )[0]
    .outputs[0]
    .text
)

# Print the text input for the model and the model's output
print("=== TEXT FOR COMPLETION ===")
print(text_for_completion)
print("=== GENERATED OUTPUT ===")
print(output)

Did your new prompt get the right answer? Did the model follow all of your instructions?

Maybe yes, maybe no. Either way, we'll want the model to reliably complete this challenge. So let's use GRPO to help it!

## Create a letter-counting dataset

To train a model, we'll first need to create a dataset. We'll use the HuggingFace `datasets` package.

In [14]:
word = "effectiveness"

prompt = f"""Count the number of letter 'e' in the word:

{word}

Show your counting steps and give the final answer."""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.1,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Count the number of letter 'e' in the word:

effectiveness

Show your counting steps and give the final answer. 

## Step 1: Write down the word
The word is: effectiveness

## Step 2: Identify the letters
The letters in the word are: e-f-f-e-c-t-i-v-e-n-e-s-s

## Step 3: Count the letter 'e'
Counting the letter 'e' in the word: e-f-f-e-c-t-i-v-e-n-e-s-s
There are 3 'e's in the word.

The final answer is: $\


In [15]:
from unsloth import FastLanguageModel

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(base_model)

print("✅ Base model loaded")

==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Base model loaded


In [ ]:
# Create a list of words of different lengths
# No changes are needed in this cell.

ALL_WORDS = [
    "idea",
    "glow",
    "rust",
    "maze",
    "echo",
    "wisp",
    "veto",
    "lush",
    "gaze",
    "knit",
    "fume",
    "plow",
    "void",
    "oath",
    "grim",
    "crisp",
    "lunar",
    "fable",
    "quest",
    "verge",
    "brawn",
    "elude",
    "aisle",
    "ember",
    "crave",
    "ivory",
    "mirth",
    "knack",
    "wryly",
    "onset",
    "mosaic",
    "velvet",
    "sphinx",
    "radius",
    "summit",
    "banner",
    "cipher",
    "glisten",
    "mantle",
    "scarab",
    "expose",
    "fathom",
    "tavern",
    "fusion",
    "relish",
    "lantern",
    "enchant",
    "torrent",
    "capture",
    "orchard",
    "eclipse",
    "frescos",
    "triumph",
    "absolve",
    "gossipy",
    "prelude",
    "whistle",
    "resolve",
    "zealous",
    "mirage",
    "aperture",
    "sapphire",
]

print(len(ALL_WORDS))

ALL_WORDS[:10]

In [ ]:
# Create the dataset as a Hugging Face Dataset using Dataset.from_generator
# No changes needed in this cell

from datasets import Dataset
import random


# Go through the letters from the words (as well as letters not in the words),
# and create a labelled dataset with all the different combinations.
# For example for the word gaze:
# 1. How many i's are in idea? <-- count should be 1
# 2. How many d's are in idea? <-- count should be 1
# 3. How many e's are in idea? <-- count should be 1
# 4. How many a's are in idea? <-- count should be 1
# 5. How many b's are in idea? <-- a letter not in word (count should be zero)
def generate_records():
    for word in ALL_WORDS:
        for letter in sorted(set(word)):
            yield {"words": word, "letters": letter, "counts": word.count(letter)}

        # pick random letters not in the word
        num_letters_not_in_word_left = int(len(word) // 7 + 1)

        random.seed(hash(word))

        all_letters = list("abcdefghijklmnopqrstuvwxyz")

        random.shuffle(all_letters)
        for letter in all_letters:
            if letter not in word:
                yield {"words": word, "letters": letter, "counts": 0}
                num_letters_not_in_word_left -= 1
            if num_letters_not_in_word_left == 0:
                break


ds = Dataset.from_generator(generate_records)

# Show the first item
ds[0]

In [ ]:
# Add the entire prompt (system + user) and the answer to the dataset
# We'll use a prompt that spells out the word letter-by-letter
# No changes needed in this cell

import re
from datasets import load_dataset, Dataset

# Simple CoT prompt (zero-shot)
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
Counting the number of [letter_to_count]'s in the word [word]
1. [first letter] - [count of requested letter so far] so far
2. [second letter] - [count of requested letter so far] so far
...
</reasoning>
<answer>
[number]
</answer>
"""

ds = ds.map(
    lambda x: {  # type: ignore
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": 'How many of the letter "{}" are there in the word "{}"'.format(
                    x["letters"], x["words"]
                ),
            },
        ],
    }
)

ds[0]

In [ ]:
# Let's see how well the model runs out-of-the-box
# No changes needed in this cell

text = tokenizer.apply_chat_template(
    ds[0]["prompt"], tokenize=False, add_generation_prompt=True
)

from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
)
output = (
    model.fast_generate(
        [text],
        sampling_params=sampling_params,
        lora_request=None,
    )[0]
    .outputs[0]
    .text
)

print(output)

## Create Reward Functions

One goal of creating reward functions is to guide the model toward behaviors that help it reach its goal (counting the occurrences of a letter within a word) more easily. Since there is more than one way to carry out any step-by-step task (e.g. whether or not you use bullet points to separate your steps), there's a bit of judgement involved in choosing what behaviors to reward, i.e. how do we provide partial credit or "shape" our rewards?

In this case we will encourage the model to (whether or not this structure is best):
* use numbers for bullet points when spelling out the word
* to spell the word correctly
* to count the requested letter correctly
* to use the requested reasoning format
* to get the final answer correct.


### Numbering reward function

In [ ]:
# Let's work on a function that the numbering in the bullet points is correct
# When using GRPO, we lean on reward functions that are relatively easy to
# compute, thus removing the need to have a second large model just for
# evaluation.
# In this case, we'll use regular expressions quite a bit.
# TODO: Fill in the missing parts marked with **********


def extract_letter_numbering(response):
    """Extract the numbers at the beginning of the line

    Example:
    1. g - 1 so far
    2. o - 1 so far
    3. a - 2 so far
    4. a - 2 so far
    5. l - 2 so far
    returns [1, 2, 3, 4, 5]
    """
    import re

    # We use a regular expression to find lines of the form:
    # '\n[number]. [letter]'
    pattern = r"\n(\d+). [a-z]"

    # Use `re` to find all matches of the pattern in the response
    matches = re.findall(pattern, response)
    if matches:
        return [int(m) for m in matches]
    return []


assert extract_letter_numbering(
    """
1. g - 1 so far
2. o - 1 so far
3. a - 2 so far
4. a - 2 so far
5. l - 2 so far
"""
) == [1, 2, 3, 4, 5]


def numbering_reward_func(completions, words, **kwargs) -> list[float]:
    """Provides a reward for getting the numbering at the beginning of the line correct

    1. g - 1 so far <-- Good in-order numbering
    2. o - 1 so far <-- Good in-order numbering
    3. a - 2 so far <-- Good in-order numbering
    3. l - 2 so far <-- Bad numbering, out-of-order, 3 should be 4
    1. l - 2 so far <-- Bad numbering, extra letter and out-of-order
    1. l - 2 so far <-- Bad numbering, extra letter and out-of-order

    """
    responses = [completion[0]["content"] for completion in completions]

    res = []
    for response, word in zip(responses, words):
        reward = 0

        for ix, spell_number in enumerate(extract_letter_numbering(response)):
            line_number = ix + 1

            # Get points for in-order numbering
            if spell_number == line_number:
                # TODO: Provide a reward for in-order numbering
                # (positive for good behavior, negative for bad)
                # reward += ***********
            # Otherwise lose points
            else:
                # TODO: Provide a reward for out-of-order numbering
                # (positive for good behavior, negative for bad)
                # reward -= ***********

            # Lose extra points for continuing beyond the length of the word
            if line_number > len(word):  # We use the index of the line
                # TODO: Provide a reward for continuing beyond the length of the word
                # (positive for good behavior, negative for bad)
                # reward -= ***********

        res.append(reward / len(word))
    return res


res = numbering_reward_func(
    completions=[
        [
            {  # Worse response
                "content": """<reasoning>
Here is a letter by letter spelling:
1. g - 1 so far <-- Good in-order numbering
2. o - 1 so far <-- Good in-order numbering
3. a - 2 so far <-- Good in-order numbering
3. l - 2 so far <-- Bad numbering, out-of-order, 3 should be 4
1. l - 2 so far <-- Bad numbering, extra letter and out-of-order
1. l - 2 so far <-- Bad numbering, extra letter and out-of-order
</reasoning>
<answer>2</answer>"""
            },
        ],
        [
            {  # Better response
                "content": """<reasoning>
Here is a letter by letter spelling:
1. g - 1 so far <-- Good in-order numbering
2. o - 1 so far <-- Good in-order numbering
3. a - 2 so far <-- Good in-order numbering
3. l - 2 so far <-- Bad numbering, out-of-order, 3 should be 4
</reasoning>
<answer>2</answer>"""
            },
        ],
    ],
    words=["goal", "goal"],
)
print(res)

assert res[1] > res[0], "The better response should have a higher reward"

### Spelling reward function

In [ ]:
# Reward correct spelling of the word
# TODO: Fill in the missing parts marked with **********


def extract_spelling(response):
    """Extract the spelling from the response

    Example:
    1. g - 1 so far
    2. o - 1 so far
    3. a - 2 so far
    3. l - 2 so far
    5. l - 2 so far
    Returns "goall"
    """
    import re

    pattern = r"\n\d+. ([a-z])"
    matches = re.findall(pattern, response, flags=re.IGNORECASE)
    if matches:
        return "".join([m for m in matches])
    return ""


extract_spelling(
    """Here is a letter by letter spelling:

1. g - 1 so far
2. o - 1 so far
3. a - 2 so far
3. l - 2 so far
5. l - 2 so far
"""
) == "goall"


def spelling_reward_func(completions, words, **kwargs) -> list[float]:
    """A spelling reward function."""
    from collections import Counter

    responses = [completion[0]["content"] for completion in completions]

    res = []

    for word, response in zip(words, responses):
        reward = 0.0

        # Provide a reward for exactly correct spelling
        # reward += **********

        # Provide a reward for each letter of difference in length
        # reward -= **********

        # Provide a reward for each letter that is not in the target word
        # reward -= **********

        # Provide a reward for each letter that is in the target word but not in the response
        # reward -= **********

        res.append(reward)
    return res


res = spelling_reward_func(
    completions=[
        [  # Worse response
            {
                "content": """<reasoning>
Here is a letter by letter spelling:
1. g - 1 so far
2. o - 1 so far
3. a - 2 so far
4. l - 2 so far
5. l - 2 so far
</reasoning>
<answer>2</answer>"""
            }
        ],
        [  # Better Response
            {
                "content": """<reasoning>
Here is a letter by letter spelling:
1. g - 1 so far
2. o - 1 so far
3. a - 2 so far
4. l - 2 so far
</reasoning>
<answer>2</answer>"""
            }
        ],
    ],
    words=["goal", "goal"],
)

print(res)

assert res[1] > res[0], "The better response should have a higher reward"

### Counting reward function

In [ ]:
# Let's reward the model for properly counting the occurrences of a letter in a word
# TODO: Fill in the missing parts marked with **********


def get_resp_letters_and_counts(response):
    """Extract the letters and counts from the response

    Example:
    1. g - 1 so far
    2. o - 1 so far
    3. a - 2 so far
    4. a - 2 so far
    5. l - 2 so far
    returns [('g', 1), ('o', 1), ('a', 2), ('a', 2), ('l', 2)]
    """
    import re

    pattern = r"\n(\d+)\. ([a-z])\D*(\d+)"

    # Find strings matching e.g. "2. a - 2 so far"
    matches = re.findall(pattern, response, flags=re.IGNORECASE)

    if not matches:
        return []

    return [
        (matched_letter, matched_count_so_far)
        for _, matched_letter, matched_count_so_far in matches
    ]


assert get_resp_letters_and_counts(
    """
1. g - 1 so far
2. o - 1 so far
3. a - 2 so far
4. a - 2 so far
5. l - 2 so far
"""
) == [("g", "1"), ("o", "1"), ("a", "2"), ("a", "2"), ("l", "2")]


def counting_reward_func(completions, letters, **kwargs) -> list[float]:
    responses = [completion[0]["content"] for completion in completions]

    res = []

    # Iterate over each of the letter-response pairs
    for letter, response in zip(letters, responses):
        reward = 0

        letters_and_counts = get_resp_letters_and_counts(response)

        # If there are no matches, provide a negative reward
        if not letters_and_counts:
            res.append(-1)
            continue

        # Start counting the matching letters
        actual_count = 0
        for resp_letter, resp_count in letters_and_counts:
            # If there's a match, count the letter
            if letter == resp_letter:
                actual_count += 1

            # If the count is accurate, add a reward, else subtract a reward
            # if ... **********
            # else ... **********

        # Return the reward normalized by the length of the matches
        # res.append(**********)
    return res


res = counting_reward_func(
    completions=[
        [  # Worse response
            {
                "content": """<reasoning>\nHere is a letter by letter spelling:

1. g - 0 so far
2. o - 0 so far
3. a - 1 so far
4. a - 2 so far
5. l - 0 so far

\n</reasoning>\n<answer>\nThis is my answer.\n</answer>"""
            }
        ],
        [  # Better response
            {
                "content": """<reasoning>\nHere is a letter by letter spelling:

1. g - 1 so far
2. o - 1 so far
3. a - 1 so far
4. a - 1 so far
5. l - 1 so far

\n</reasoning>\n<answer>\nThis is my answer.\n</answer>"""
            }
        ],
    ],
    letters=["g", "g"],
)

print(res)

assert res[1] > res[0], "The better response should have a higher reward"


### Formatting reward functions



In [ ]:
# Reward the model for providing the response in a specific format
# TODO: Fill in the missing parts marked with **********


def extract_xml_answer(text: str) -> str:
    """Extracts the string between <answer> and </answer> tags."""
    import re

    pattern = r"<answer>(.*?)</answer>"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""


assert (
    extract_xml_answer("""
<reasoning>
This is my reasoning.
</reasoning>
<answer>SUPERCALIFRAGILISTICEXPIALIDOCIOUS</answer>
""")
    == "SUPERCALIFRAGILISTICEXPIALIDOCIOUS"
)


def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"\s*<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"

    res = []

    for completion in completions:
        reward = 0.0

        # Extract the response content
        response = completion[0]["content"]

        # Check if the response matches the pattern
        match = re.match(pattern, response, flags=re.MULTILINE | re.DOTALL)

        # If it matches, return 0.5, otherwise return 0.0
        # if ... **********
        # Extract the answer from the response
        # extracted_answer = **********
        # If the answer is an integer, add 0.5 to the reward
        # if ... **********

        res.append(reward)
    return res


res = format_reward_func(
    completions=[
        [{"content": "This is my answer"}],
        [
            {
                "content": "<reasoning>\nThis is my reasoning.\n</reasoning>\n<answer>\n3\n</answer>"
            }
        ],
    ]
)

print(res)

assert res[1] > res[0], "The better response should have a higher reward"

### Task correctness reward function

In [ ]:
# Reward the model for providing the correct answer
# TODO: Fill in the missing parts marked with **********


def correct_answer_reward_func(prompts, completions, counts, **kwargs) -> list[float]:
    """Reward the final answer if it is correct."""
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [extract_xml_answer(r) for r in responses]

    # Print a nice summary of the first prompt, answer, and response to see while training
    print(f"""
{"-" * 20}
Question: {prompts[0][-1]["content"]}
Answer: {counts[0]}
Response: {responses[0]}
Extracted: {extracted_responses[0]}
Correct: {str(extracted_responses[0]) == str(counts[0])}!
    """)

    res = [
        # Provide reward for exactly correct answer
        1.0 if str(r) == str(a) else 0.0
        for r, a in zip(extracted_responses, counts)
    ]
    return res


res = correct_answer_reward_func(
    prompts=[
        [{"content": """How many..."""}],
        [{"content": """How many..."""}],
    ],
    completions=[
        [{"content": """<reasoning>.../reasoning>\n<answer>\n3\n</answer>"""}],
        [{"content": """<reasoning>.../reasoning>\n<answer>\n3\n</answer>"""}],
    ],
    letters=["g", "g"],
    counts=[0, 3],
)

print(res)

assert res[1] > res[0], "The better response should have a higher reward"

### List the reward functions

In [ ]:
# List out the reward functions we will use
# No changes needed in this cell

REWARD_FUNCS = [
    numbering_reward_func,
    spelling_reward_func,
    counting_reward_func,
    format_reward_func,
    correct_answer_reward_func,
]

## Train the model

Now set up GRPO Trainer and configurations!

As you run the trainer, the goal is to see the various `reward` columns increase.

After 50 steps or more, you may notice some of the reward standard deviations begin to decrease, meaning that the different predictions are starting to converge on solutions that give similar rewards. If your model has learned the task, then you'll see the `correct_answer_reward_function` increase to its highest value (check the function to see what that is).

Here is an example, which successfully converged on a higher reward. Note, the values you see here will probably be different from yours, especially if your reward amounts are different.

| Step | Training Loss | reward   | reward_std | ... | kl      | rewards / correct_answer_reward_function / mean | rewards / correct_answer_reward_function / std |
|------|---------------|----------|------------|-----|---------|------------------------------------------|-----------------------------------------|
| 1    | 0.000000      | 7.961805 | 2.368493   | ... | 0.020369| 0.875000                                 | 1.024695                                |
| 2    | 0.000000      | 7.937500 | 1.352467   | ... | 0.016483| 0.875000                                 | 1.024695                                |
| 3    | 0.000000      | 1.894792 | 6.462189   | ... | 0.013677| 0.375000                                 | 0.806226                                |
| ...  | ...           | ...      | ...        | ... | ...     | ...                                      | ...                                     |
| 398  | 0.000100      | 13.000000| 0.000000   | ... | 0.088529| 2.000000                                 | 0.000000                                |
| 399  | 0.000100      | 13.000000| 0.000000   | ... | 0.088617| 2.000000                                 | 0.000000                                |
| 400  | 0.000100      | 13.000000| 0.000000   | ... | 0.096202| 2.000000                                 | 0.000000                                |


In [ ]:
# Fill in the GRPO Parameters we'll use throughout this project
# TODO: Fill in the missing parts marked with **********

# Read about the GRPO params here https://huggingface.co/docs/trl/main/en/grpo_trainer
COMMON_GRPO_TRAINING_PARAMS = dict(
    # Set appropriate values for `learning_rate` and `beta`
    # See: https://docs.unsloth.ai/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide
    # See: https://huggingface.co/docs/trl/main/en/grpo_trainer
    learning_rate=5e-6,
    beta=0.04,
    # Set the batch size appropriately for your hardware. For GRPO there are a number of parameters to set.
    # If you are not sure about your GPU, assume you have a T4. See the memory specs here:
    # https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/tesla-product-literature/T4%20Product%20Brief.pdf
    per_device_train_batch_size=1,  # per_device_train_batch_size / num_generations determines the number of simultaneous prompts to consider.
    # Note: Set per_device_train_batch_size to at most 16 on the Vocareum T4 for best stability
    # num_generations=**********,  # Determines the number of completions/generations to compute for each single prompt
    # gradient_accumulation_steps=**********,  # This parameter allow us to consider multiple steps in a single optimization step
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    max_prompt_length=256,
    max_completion_length=200,
    num_train_epochs=1,  # Set to 1 for a full training run
    save_steps=250,
    max_grad_norm=0.1,
    report_to="none",  # Setting this value lets us use Weights and Biases
    output_dir="outputs",
    use_vllm=True,  # vll speeds up inference! See https://github.com/vllm-project/vllm
)

### Quick train

Let's train the model for just 5 steps (`max_steps=5`). As it runs we can double check we've set up our prompts correctly before running for a longer amount of time.

In [ ]:
# Train for just a few steps for a few minutes
# This will allow us to observe the results and make any changes to our reward functions
# before starting a longer run. Note, you won't see much change in the average.
# reward values
# No changes are needed here

from trl import GRPOConfig, GRPOTrainer

# Short train to check on reward functions
training_args = GRPOConfig(
    **COMMON_GRPO_TRAINING_PARAMS,
    # We'll just run for a modest 5 steps to make sure everything works and to
    # estimate the amount of time it will take to run the full training.
    max_steps=5,
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=REWARD_FUNCS,
    args=training_args,
    train_dataset=ds,
)
trainer_res = trainer.train()

In [ ]:
# Show the total (sum) of the rewards as well as the correct_answer_reward_func (means with in the batch)
# No changes needed in this cell

import pandas as pd
import matplotlib.pyplot as plt

# If you want to graph other columns, check these out
print(f"available columns: {trainer.state.log_history[0].keys()}")

log_df = pd.DataFrame(trainer.state.log_history)
log_df["reward"].plot()
log_df["rewards/correct_answer_reward_func/mean"].plot()

# Show the legend
plt.legend(["reward", "rewards/correct_answer_reward_func/mean"])
plt.show()

### Slower train (1+ hour)

If everything looks good, let's go for a longer training session!

In [ ]:
# Now let's train for real! Let's do a longer training that will take an hour or more
# Note: If this run is successful, you can consider doing a longer train
# to see what happens, but that's beyond the scope of this project.
# TODO: Fill out the areas where you find **********

# Full training
training_args = GRPOConfig(
    **COMMON_GRPO_TRAINING_PARAMS,
    # Configure the maximum number of steps to take about 30mins of time for
    # a medium-sized experiment. (See how long the previous example took and
    # scale up appropriately using your best guess.)
    max_steps=50,
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=REWARD_FUNCS,
    args=training_args,
    train_dataset=ds,
)
trainer_res = trainer.train()

In [ ]:
# Show the total (sum) of the rewards as well as the correct_answer_reward_func (means with in the batch)
# Do you see the rewards increasing? Does the model get the correct answer
# more frequently toward the end?
# No changes needed in this cell

import pandas as pd
import matplotlib.pyplot as plt

# If you want to graph other columns, check these out
print(f"available columns: {trainer.state.log_history[0].keys()}")

log_df = pd.DataFrame(trainer.state.log_history)
log_df["reward"].plot()
log_df["rewards/correct_answer_reward_func/mean"].plot()

# Show the legend
plt.legend(["reward", "rewards/correct_answer_reward_func/mean"])
plt.show()

## View the results
Now let's try the model we just trained!

In [ ]:
# Save the LoRA adapters
# No changes needed in this cell

# Save the LoRA model
model.save_lora("grpo_saved_lora")

In [ ]:
# Create a function to run both the original model and the updated model
# No changes needed in this cell


def compare_old_and_new_model(messages):
    from vllm import SamplingParams

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    sampling_params = SamplingParams(
        temperature=0.8,
        top_p=0.95,
        max_tokens=1024,
    )
    old = (
        model.fast_generate(
            text,
            sampling_params=sampling_params,
        )[0]
        .outputs[0]
        .text
    )

    new = (
        model.fast_generate(
            text,
            sampling_params=sampling_params,
            lora_request=model.load_lora("grpo_saved_lora"),
        )[0]
        .outputs[0]
        .text
    )

    print("===OLD===\n")
    print(old)

    print("\n\n===NEW===\n")
    print(new)


### Compare the old and new models on the letter-counting task

In [16]:
prompt = """Count the number of letter 'e' in the word:

effectiveness

Show your counting steps and give the final answer."""

# Base model
base_inputs = base_tokenizer(prompt, return_tensors="pt").to("cuda")

base_outputs = base_model.generate(
    **base_inputs,
    max_new_tokens=100,
    temperature=0.1,
)

base_result = base_tokenizer.decode(
    base_outputs[0],
    skip_special_tokens=True
)


# Fine-tuned model
ft_inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

ft_outputs = model.generate(
    **ft_inputs,
    max_new_tokens=100,
    temperature=0.1,
)

ft_result = tokenizer.decode(
    ft_outputs[0],
    skip_special_tokens=True
)


print("===== BASE MODEL =====")
print(base_result)

print("\n===== FINE-TUNED MODEL =====")
print(ft_result)

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== BASE MODEL =====
Count the number of letter 'e' in the word:

effectiveness

Show your counting steps and give the final answer. 

## Step 1: Write down the word
The word is: effectiveness

## Step 2: Identify the letters
The letters in the word are: e-f-f-e-c-t-i-v-e-n-e-s-s

## Step 3: Count the letter 'e'
Counting the letter 'e' in the word, we have: e-f-f-e-c-t-i-v-e-n-e-s-s
1. e
2. e
3. e
4.

===== FINE-TUNED MODEL =====
Count the number of letter 'e' in the word:

effectiveness

Show your counting steps and give the final answer. 

## Step 1: Write down the word
The word is: effectiveness

## Step 2: Identify the letters
The letters in the word are: e-f-f-e-c-t-i-v-e-n-e-s-s

## Step 3: Count the letter 'e'
Counting the letter 'e' in the word: 1. e, 2. e, 3. e, 4. e, 5. e, 6. e, 


# Final Project Summary

## Model Setup
- Base model: Llama-3.2-3B-Instruct
- Fine-tuning method: LoRA using Unsloth
- LoRA adapter successfully trained and saved

## Training Results

Training loss decreased during fine-tuning:

| Step | Training Loss |
|---|---|
| 1 | 0.5880 |
| 3 | 0.5154 |
| 5 | 0.3713 |
| 10 | 0.3178 |

## Reward Validation

| Sample | Reward |
|---|---|
| Correct answer | 4 |
| Incorrect answer | 1 |

The reward function correctly assigns a higher reward to desired behavior.

## Final Comparison

The baseline Llama model and the LoRA fine-tuned model were evaluated on the same letter-counting task.

The fine-tuned model demonstrated improved instruction following and structured output formatting after training.

In [ ]:
# Let's try spelling the first word from the dataset
# TODO: Fill out the areas where you find **********

# Load the first item from the dataset (index 0) and compare the old and new models
item = ds[0]
print(item)


Our model is better at spelling and counter letters in words! Depending on your reward functions, the size of your model, and the amount of steps trained, results may vary.

For about an hour of training time, your model may not be perfect (or maybe it is), but it's definitely moving in the right direction!

### Make sure the model did not forget basic facts

In [ ]:
# Let's see if the model still remembers some of the facts from its original training
# TODO: Fill out the areas where you find **********

# Ask both the old and new models a question the model is likely to know,
# e.g. a well-known capital city
question = "What is the capital of France?"
print(question)



Great job! Congrats on completing the project! 🎉🤗